
ideas:
1. Conocer los clientes que tienen cambios a digital para modelo productivo.
2. Preguntas de fondo ¿porque? y ¿para que?

# Analisis inicial

In [0]:
# Cargar la tabla en un DataFrame de **Spark**
df = spark.table("data.client_transaction_orders")

In [0]:
from pyspark.sql import functions as F

df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()


In [0]:
# Muestra las primeras 10 filas en formato de tabla interactiva
display(df.limit(10))

In [0]:
# Contar el número total de filas en el DataFrame
total_filas = df.count()

print(f"El número total de filas es: {total_filas}")

# Contar el número de clientes únicos
total_clientes_unicos = df.select("cliente_id").distinct().count()

print(f"El número total de clientes únicos es: {total_clientes_unicos}")

In [0]:
from pyspark.sql.functions import min, max

# Calcula la fecha mínima y máxima en una sola operación
rango_fechas = df.agg(
    min("fecha_pedido_dt").alias("primera_fecha_pedido"),
    max("fecha_pedido_dt").alias("ultima_fecha_pedido")
)

# Muestra el resultado
display(rango_fechas)

In [0]:
from pyspark.sql.functions import quarter, year, col, count

# Añade una columna de año y trimestre para el análisis temporal
df_temporal = df.withColumn("trimestre", quarter(col("fecha_pedido_dt"))) \
                .withColumn("año", year(col("fecha_pedido_dt")))

# Agrupa por año/trimestre y canal para ver la evolución
evolucion_canal = df_temporal.groupBy("año", "trimestre", "canal_pedido_cd") \
    .agg(count("*").alias("numero_de_pedidos")) \
    .orderBy("año", "trimestre")

display(evolucion_canal)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# Obtenemos la última transacción de cada cliente
windowSpec = Window.partitionBy("cliente_id").orderBy(desc("fecha_pedido_dt"))
df_ultima_transaccion = df.withColumn("rank", row_number().over(windowSpec)).filter("rank = 1")

# Filtramos para quedarnos con aquellos cuya última transacción NO fue digital
ultimas_no_digitales = df_ultima_transaccion.filter(col("canal_pedido_cd") != "DIGITAL") \
    .select("cliente_id")

# Unimos con la tabla RFM y filtramos por recencia (ej. compraron en los últimos 90 días)
clientes_para_convertir = rfm_table.join(ultimas_no_digitales, on="cliente_id") \
    .filter(col("recencia") <= 90) \
    .orderBy("recencia")

print(f"Clientes recientes para convertir a digital: {clientes_para_convertir.count()}")
display(clientes_para_convertir)

## Porcentaje de cliente digitales, no digitales y multicanal

In [0]:
from pyspark.sql.functions import collect_set, when, col, count, lit, size, array_contains

# 1. Agrupa por cliente para obtener el conjunto de canales que ha usado cada uno
canales_por_cliente = df.groupBy("cliente_id") \
    .agg(collect_set("canal_pedido_cd").alias("canales_usados"))

# 2. Clasifica a cada cliente en una categoría
clasificacion_clientes = canales_por_cliente.withColumn(
    "segmento_canal",
    when(
        size(col("canales_usados")) > 1, lit("Multicanal")
    ).when(
        array_contains(col("canales_usados"), "DIGITAL"), lit("Solo Digital") # CORRECCIÓN: Usar array_contains
    ).otherwise(
        lit("Solo No Digital")
    )
)

# 3. Cuenta el número total de clientes únicos
total_clientes = df.select("cliente_id").distinct().count()

# 4. Cuenta los clientes por cada segmento y calcula el porcentaje
resumen_segmentos = clasificacion_clientes.groupBy("segmento_canal") \
    .agg(count("*").alias("numero_de_clientes")) \
    .withColumn(
        "porcentaje",
        (col("numero_de_clientes") / total_clientes) * 100
    )

# Muestra el resultado final
display(resumen_segmentos)

 # 1. ¿Son Diferentes los Pedidos Digitales? 
[](url)

In [0]:
from pyspark.sql.functions import avg, sum, median, count, countDistinct, col

# Agrupamos por canal y calculamos un resumen completo de las variables numéricas
analisis_por_canal = df.groupBy("canal_pedido_cd") \
    .agg(
        count("*").alias("total_pedidos"),
        countDistinct("cliente_id").alias("clientes_unicos"),
        avg("facturacion_usd_val").alias("facturacion_promedio"),
        median("facturacion_usd_val").alias("facturacion_mediana"),
        sum("facturacion_usd_val").alias("facturacion_total"),
        avg("materiales_distintos_val").alias("promedio_materiales_distintos"),
        median("materiales_distintos_val").alias("mediana_materiales_distintos"),
        avg("cajas_fisicas").alias("promedio_cajas_fisicas"),
        median("cajas_fisicas").alias("mediana_cajas_fisicas")
    )
# Añade la nueva columna "pedidos_por_cliente" al DataFrame anterior
analisis_por_canal = analisis_por_canal.withColumn(
    "pedidos_por_cliente",
    col("total_pedidos") / col("clientes_unicos")
)

# Muestra la tabla de resultados
display(analisis_por_canal)


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Seleccionamos las columnas necesarias para el gráfico
display(df.select("canal_pedido_cd", "facturacion_usd_val","cajas_fisicas","materiales_distintos_val"))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count

# Agrupamos por país y por canal para contar el número de pedidos
resumen_pais = df.groupBy("pais_cd", "canal_pedido_cd") \
    .agg(count("*").alias("numero_de_pedidos")) \
    .orderBy("pais_cd", "canal_pedido_cd")

display(resumen_pais)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count

# Agrupamos por región y por canal
resumen_region = df.groupBy("region_comercial_txt", "canal_pedido_cd") \
    .agg(count("*").alias("numero_de_pedidos")) \
    .orderBy("region_comercial_txt", "canal_pedido_cd")

display(resumen_region)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import count

# Agrupamos por tipo de cliente y por canal
resumen_tipo_cliente = df.groupBy("tipo_cliente_cd", "canal_pedido_cd") \
    .agg(count("*").alias("numero_de_pedidos")) \
    .orderBy("tipo_cliente_cd", "canal_pedido_cd")

display(resumen_tipo_cliente)

Databricks visualization. Run in Databricks to view.

# 2. ¿Quiénes son los Clientes Digitales? (Atributos del Cliente)


In [0]:
from pyspark.sql.functions import collect_set, when, col, count, lit, size, array_contains

# 1. Agrupa por cliente para obtener el conjunto de canales que ha usado cada uno
canales_por_cliente = df.groupBy("cliente_id") \
    .agg(collect_set("canal_pedido_cd").alias("canales_usados"))

# 2. Clasifica a cada cliente en una categoría
clasificacion_clientes = canales_por_cliente.withColumn(
    "segmento_canal",
    when(
        size(col("canales_usados")) > 1, lit("Multicanal")
    ).when(
        array_contains(col("canales_usados"), "DIGITAL"), lit("Solo Digital") # CORRECCIÓN: Usar array_contains
    ).otherwise(
        lit("Solo No Digital")
    )
)

# 3. Cuenta el número total de clientes únicos
total_clientes = df.select("cliente_id").distinct().count()

# 4. Cuenta los clientes por cada segmento y calcula el porcentaje
resumen_segmentos = clasificacion_clientes.groupBy("segmento_canal") \
    .agg(count("*").alias("numero_de_clientes")) \
    .withColumn(
        "porcentaje",
        (col("numero_de_clientes") / total_clientes) * 100
    )

# Muestra el resultado final
display(resumen_segmentos)

2. ¿Quiénes son los Clientes Digitales? (Atributos del Cliente)


Se encontro que la uncia variable que parece ser diferencia entre los que usan el canal digital almenos 1 veces es solo el nivel de madurez digital, el resto de varibels, como pais, region, tipo de cliente... no son muestran diferencai significativas

In [0]:
from pyspark.sql.functions import min, col

# 1. Encuentra la primera fecha de pedido digital para cada cliente
primer_pedido_digital = df.filter(col("canal_pedido_cd") == "DIGITAL") \
    .groupBy("cliente_id") \
    .agg(min("fecha_pedido_dt").alias("primera_fecha_digital"))

# 2. Encuentra todos los pedidos no digitales
pedidos_no_digitales = df.filter(col("canal_pedido_cd") != "DIGITAL")

# 3. Une las dos tablas. Buscamos clientes que tengan un pedido no digital ANTES de su primer pedido digital.
clientes_que_cambiaron = pedidos_no_digitales.join(
    primer_pedido_digital,
    on="cliente_id"
).filter(
    col("fecha_pedido_dt") < col("primera_fecha_digital")
)

# 4. Cuenta el número de clientes únicos que cumplen la condición
numero_de_cambios = clientes_que_cambiaron.select("cliente_id").distinct().count()

print(f"El número de clientes que cambiaron de un canal no digital a digital es: {numero_de_cambios}")

In [0]:
from pyspark.sql.functions import min, col, when, lit, countDistinct
from pyspark.sql.functions import when, col, countDistinct, lit

# 1. Crea un DataFrame con los IDs de los clientes digitales y les añade una columna literal
clientes_digitales_ids = df.filter(col("canal_pedido_cd") == "DIGITAL") \
    .select("cliente_id").distinct() \
    .withColumn("usa_canal_digital", lit("Si")) 

# 2. Etiqueta a cada cliente en el DataFrame original
df_etiquetado = df.join(clientes_digitales_ids, on="cliente_id", how="left") \
    .withColumn(
        "usa_canal_digital", 
        when(col("usa_canal_digital") == "Si", lit("Si")).otherwise(lit("No")) # Usar lit() aquí también es una buena práctica
    )
# --- CÓDIGO PARA IDENTIFICAR SWITCHERS (el que ya tenías) ---
primer_pedido_digital = df.filter(col("canal_pedido_cd") == "DIGITAL") \
    .groupBy("cliente_id") \
    .agg(min("fecha_pedido_dt").alias("primera_fecha_digital"))

pedidos_no_digitales = df.filter(col("canal_pedido_cd") != "DIGITAL")

clientes_que_cambiaron_ids = pedidos_no_digitales.join(
    primer_pedido_digital,
    on="cliente_id"
).filter(
    col("fecha_pedido_dt") < col("primera_fecha_digital")
).select("cliente_id").distinct().withColumn("segmento", lit("Switcher"))
# --- FIN DEL CÓDIGO DE IDENTIFICACIÓN ---

# Unimos esta lista a nuestro DataFrame etiquetado para crear el segmento
df_segmentado = df_etiquetado.join(
    clientes_que_cambiaron_ids,
    on="cliente_id",
    how="left"
).withColumn(
    "segmento",
    when(col("segmento") == "Switcher", "Switcher").otherwise("No Switcher")
)

In [0]:
# Agrupamos por madurez y el nuevo segmento
perfil_switchers_madurez = df_segmentado.groupBy("madurez_digital_cd", "segmento") \
    .agg(countDistinct("cliente_id").alias("numero_de_clientes")) \
    .orderBy("madurez_digital_cd", "segmento")

display(perfil_switchers_madurez)

Databricks visualization. Run in Databricks to view.

In [0]:
# Agrupamos por frecuencia y el nuevo segmento
perfil_switchers_frecuencia = df_segmentado.groupBy("frecuencia_visitas_cd", "segmento") \
    .agg(countDistinct("cliente_id").alias("numero_de_clientes")) \
    .orderBy("frecuencia_visitas_cd", "segmento")

display(perfil_switchers_frecuencia)

Databricks visualization. Run in Databricks to view.

In [0]:
# Agrupamos por tipo_cliente_cd y el segmento
perfil_switchers_tipo = df_segmentado.groupBy("tipo_cliente_cd", "segmento") \
    .agg(countDistinct("cliente_id").alias("numero_de_clientes")) \
    .orderBy("tipo_cliente_cd", "segmento")

display(perfil_switchers_tipo)

Databricks visualization. Run in Databricks to view.

In [0]:


# 3. Agrupa por el atributo a analizar y la etiqueta
perfil_por_madurez = df_etiquetado.groupBy(
    col("madurez_digital_cd"), 
    col("usa_canal_digital")
).agg(
    countDistinct("cliente_id").alias("numero_de_clientes")
).orderBy(
    col("madurez_digital_cd"), 
    col("usa_canal_digital")
)

# Muestra el resultado
display(perfil_por_madurez)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, countDistinct

# Agrupamos por tipo_cliente_cd y la etiqueta digital
perfil_por_tipo = df_etiquetado.groupBy(
    col("tipo_cliente_cd"), 
    col("usa_canal_digital")
).agg(
    countDistinct("cliente_id").alias("numero_de_clientes")
).orderBy(
    col("tipo_cliente_cd"), 
    col("usa_canal_digital")
)

# Muestra el resultado
display(perfil_por_tipo)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, countDistinct

# Agrupamos por estrellas_txt y la etiqueta digital
perfil_por_estrellas = df_etiquetado.groupBy(
    col("estrellas_txt"), 
    col("usa_canal_digital")
).agg(
    countDistinct("cliente_id").alias("numero_de_clientes")
).orderBy(
    col("estrellas_txt"), 
    col("usa_canal_digital")
)

# Muestra el resultado
display(perfil_por_estrellas)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, countDistinct

# Agrupamos por pais_cd y la etiqueta digital
perfil_por_pais = df_etiquetado.groupBy(
    col("pais_cd"), 
    col("usa_canal_digital")
).agg(
    countDistinct("cliente_id").alias("numero_de_clientes")
).orderBy(
    col("pais_cd"), 
    col("usa_canal_digital")
)

display(perfil_por_pais)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, countDistinct

# Agrupamos por region_comercial_txt y la etiqueta digital
perfil_por_region = df_etiquetado.groupBy(
    col("region_comercial_txt"), 
    col("usa_canal_digital")
).agg(
    countDistinct("cliente_id").alias("numero_de_clientes")
).orderBy(
    col("region_comercial_txt"), 
    col("usa_canal_digital")
)

display(perfil_por_region)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, countDistinct, desc

# 1. Contamos el total de clientes únicos por agencia
total_clientes_agencia = df_etiquetado.groupBy("agencia_id") \
    .agg(countDistinct("cliente_id").alias("total_clientes"))

# 2. Contamos los clientes digitales únicos por agencia
clientes_digitales_agencia = df_etiquetado.filter(col("usa_canal_digital") == "Si") \
    .groupBy("agencia_id") \
    .agg(countDistinct("cliente_id").alias("clientes_digitales"))

# 3. Unimos las dos tablas y calculamos la tasa de adopción
tasa_adopcion_agencia = total_clientes_agencia.join(
    clientes_digitales_agencia,
    on="agencia_id",
    how="left"
).na.fill(0) # Rellenamos con 0 las agencias sin clientes digitales
    
tasa_adopcion_agencia = tasa_adopcion_agencia.withColumn(
    "tasa_adopcion_digital",
    (col("clientes_digitales") / col("total_clientes")) * 100
)

# 4. Mostramos el top 10 de agencias con mayor adopción
top_10_agencias = tasa_adopcion_agencia.orderBy(desc("tasa_adopcion_digital")).limit(10)

display(top_10_agencias)

In [0]:
from pyspark.sql.functions import col, countDistinct, desc

# 1. Contamos el total de clientes únicos por ruta
total_clientes_ruta = df_etiquetado.groupBy("ruta_id") \
    .agg(countDistinct("cliente_id").alias("total_clientes"))

# 2. Contamos los clientes digitales únicos por ruta
clientes_digitales_ruta = df_etiquetado.filter(col("usa_canal_digital") == "Si") \
    .groupBy("ruta_id") \
    .agg(countDistinct("cliente_id").alias("clientes_digitales"))

# 3. Unimos las dos tablas y calculamos la tasa de adopción
tasa_adopcion_ruta = total_clientes_ruta.join(
    clientes_digitales_ruta,
    on="ruta_id",
    how="left"
).na.fill(0) # Rellenamos con 0 las rutas sin clientes digitales
    
tasa_adopcion_ruta = tasa_adopcion_ruta.withColumn(
    "tasa_adopcion_digital",
    (col("clientes_digitales") / col("total_clientes")) * 100
)

# 4. Mostramos el top 10 de rutas con mayor adopción
top_10_rutas = tasa_adopcion_ruta.orderBy(desc("tasa_adopcion_digital")).limit(10)

display(top_10_rutas)

# 3. ¿Cómo se Comportan los Clientes Digitales? (Historial y Frecuencia)

In [0]:
from pyspark.sql.functions import col, countDistinct

# Agrupamos por la frecuencia de visita y la etiqueta digital
perfil_por_frecuencia = df_etiquetado.groupBy(
    col("frecuencia_visitas_cd"), 
    col("usa_canal_digital")
).agg(
    countDistinct("cliente_id").alias("numero_de_clientes")
).orderBy(
    col("frecuencia_visitas_cd"), 
    col("usa_canal_digital")
)

# Muestra el resultado
display(perfil_por_frecuencia)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import max, col, datediff

# 1. Primero, encontramos la fecha más reciente en todo el dataset para usarla como referencia
fecha_mas_reciente = df.agg(max("fecha_pedido_dt")).collect()[0][0]

# 2. Ahora, para cada cliente, encontramos la fecha de su última compra
ultima_compra_por_cliente = df_etiquetado.groupBy(
    col("cliente_id"), 
    col("usa_canal_digital")
).agg(
    max("fecha_pedido_dt").alias("ultima_fecha_compra")
)

# 3. Calculamos la recencia en días
# (la diferencia entre la fecha de referencia y la última compra de cada cliente)
recencia_calculada = ultima_compra_por_cliente.withColumn(
    "recencia_en_dias",
    datediff(
        lit(fecha_mas_reciente),
        col("ultima_fecha_compra")
    )
)

# Muestra la tabla final para que puedas graficarla
display(recencia_calculada)

Databricks visualization. Run in Databricks to view.

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import sum

# 1. Calcular la facturación total por cliente (si no lo has hecho)
facturacion_por_cliente = df_no_digital.groupBy("cliente_id") \
    .agg(sum("facturacion_usd_val").alias("facturacion_total_cliente"))

# 2. Calcular los umbrales para el top 30%, 20% y 10%
umbrales = facturacion_por_cliente.approxQuantile(
    "facturacion_total_cliente", [0.70, 0.80, 0.90], 0.01
)
umbral_top_30 = umbrales[0] # Percentil 70
umbral_top_20 = umbrales[1] # Percentil 80
umbral_top_10 = umbrales[2] # Percentil 90

# 3. Convertir a pandas para el gráfico
pandas_clientes_df = facturacion_por_cliente.toPandas()

# 4. Crear el histograma
sns.set_style("whitegrid")
plt.figure(figsize=(12, 7))
sns.histplot(data=pandas_clientes_df, x='facturacion_total_cliente', bins=50, kde=True)

# 5. Añadir las líneas de percentiles superiores al gráfico
plt.axvline(x=umbral_top_30, color='gold', linestyle='--', label=f'Top 30% (desde ${umbral_top_30:,.2f})')
plt.axvline(x=umbral_top_20, color='darkorange', linestyle='--', label=f'Top 20% (desde ${umbral_top_20:,.2f})')
plt.axvline(x=umbral_top_10, color='red', linestyle='--', label=f'Top 10% (desde ${umbral_top_10:,.2f})')

# Añadir títulos, etiquetas y la leyenda
plt.title('Distribución de Facturación por Cliente no digitales con Umbrales Superiores', fontsize=16)
plt.xlabel('Facturación Total por Cliente (USD)', fontsize=12)
plt.ylabel('Número de Clientes', fontsize=12)
plt.legend() # Muestra las etiquetas de las líneas

# Muestra el gráfico
plt.show()